# Notebook Setup

In [5]:
import sys
import os
from pathlib import Path

# Path to your project root
project_dir = Path(r"C:\Users\dmika\DEV\Projects-local\dp100-learn")

# Change the working directory
os.chdir(project_dir)

# Add to sys.path if not already there
if str(project_dir) not in sys.path:
    sys.path.insert(0, str(project_dir))

In [11]:
# Now you can import from utils
from utils.consts import SUBSCRIPTION_ID, PREFERED_RESOURCE_LOCATION, MAIN_STORAGE_ACCOUNT_ACCESS_KEY

import numpy as np
import pandas as pd

from azure.identity import DefaultAzureCredential
from azure.ai.ml import MLClient

subscription_id = SUBSCRIPTION_ID
azure_credentials = DefaultAzureCredential()

resource_group_name = "ml-workspace-dev"
resource_group_location = PREFERED_RESOURCE_LOCATION

storage_account_name = "dmpdp100storageaccount99"  # must be globally unique
storage_account_location = PREFERED_RESOURCE_LOCATION
storage_container_name = "dmpdp100data"
storage_account_access_key = MAIN_STORAGE_ACCOUNT_ACCESS_KEY

azureml_workspace_name = "mlw-dp100-labs"
azureml_resource_location = PREFERED_RESOURCE_LOCATION

datastore_name = "dmdp100datastore"


ml_client = MLClient(
    credential=azure_credentials,
    subscription_id=subscription_id,
    resource_group_name=resource_group_name,
    workspace_name=azureml_workspace_name
)

# Prepare Data

## Download and extract the data locally

In [6]:
dataset_dir = Path(os.path.join(project_dir, "data/intel-image-classification"))
subset_dir = dataset_dir / "subset"

In [7]:
# Download and unzip dataset from Kaggle
import kaggle
kaggle.api.authenticate()

dataset_dir = Path(os.path.join(project_dir, "data/intel-image-classification"))
dataset_dir.mkdir(parents=True, exist_ok=True)

kaggle.api.dataset_download_files('puneet6060/intel-image-classification', path=dataset_dir, unzip=True)

Dataset URL: https://www.kaggle.com/datasets/puneet6060/intel-image-classification


## Manipulate the data

In [8]:
# Flatten directory structure
import shutil

base = dataset_dir
for folder in ["seg_train", "seg_test", "seg_pred"]:
    inner = base / folder / folder
    if inner.exists():
        for item in inner.iterdir():
            shutil.move(str(item), str(base / folder))
        shutil.rmtree(inner)

## Create a subset

In [38]:
# Take a subset of the training data for quicker experiments
import random

subset_dir = dataset_dir / "subset"
subset_dir.mkdir(exist_ok=True)
subset_dir = subset_dir / "data"
subset_dir.mkdir(exist_ok=True)

for class_dir in (dataset_dir / "seg_train").iterdir():
    if class_dir.is_dir():
        files = list(class_dir.glob("*.jpg"))
        sample_files = random.sample(files, min(200, len(files)))
        target_dir = subset_dir / class_dir.name
        target_dir.mkdir(exist_ok=True)
        for f in sample_files:
            shutil.copy(f, target_dir / f.name)
print(f"Subset created at: {subset_dir}")

Subset created at: C:\Users\dmika\DEV\Projects-local\dp100-learn\data\intel-image-classification\subset\data


## Create a data asset

In [39]:
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes

my_data = Data(
    path=str(subset_dir),
    datastore=datastore_name,
    type=AssetTypes.URI_FOLDER,
    description="Subset of Intel Image Classification dataset for AutoML image training",
    name="intel-image-subset-folder",
)
uploaded_data  = ml_client.data.create_or_update(my_data)

Uploading data (18.22 MBs):   0%|          | 75782/18221626 [00:00<00:25, 716909.87it/s]2025-10-15 18:01:04,708 WARNING Connection pool is full, discarding connection: dmpdp100storageaccount99.blob.core.windows.net. Connection pool size: 9
2025-10-15 18:01:04,708 WARNING Connection pool is full, discarding connection: dmpdp100storageaccount99.blob.core.windows.net. Connection pool size: 9
2025-10-15 18:01:04,715 WARNING Connection pool is full, discarding connection: dmpdp100storageaccount99.blob.core.windows.net. Connection pool size: 9
2025-10-15 18:01:04,708 WARNING Connection pool is full, discarding connection: dmpdp100storageaccount99.blob.core.windows.net. Connection pool size: 9
2025-10-15 18:01:04,715 WARNING Connection pool is full, discarding connection: dmpdp100storageaccount99.blob.core.windows.net. Connection pool size: 9
2025-10-15 18:01:04,715 WARNING Connection pool is full, discarding connection: dmpdp100storageaccount99.blob.core.windows.net. Connection pool size: 9


In [42]:
img_data_asset = ml_client.data.get("intel-image-subset-folder", version="1")
img_data_asset_base_path = img_data_asset.path

## Create annotation JSONL file 

### Cloud path

In [44]:
import json

img_data_asset = ml_client.data.get("intel-image-subset-folder", version="1")
base_uri = img_data_asset.path
annotations_dir = subset_dir.parent / "annotations"
annotations_dir.mkdir(exist_ok=True)
jsonl_path = annotations_dir / "train_annotations.jsonl"
records = []

for class_dir in subset_dir.iterdir():
    if class_dir.is_dir():
        label = class_dir.name
        for img in class_dir.glob("*.jpg"):
            rel = img.relative_to(subset_dir)
            records.append({"image_url": f"{base_uri}/{rel.as_posix()}", "label": label})

with open(jsonl_path, "w", encoding="utf-8") as f:
    for r in records:
        f.write(json.dumps(r) + "\n")

print("✅ JSONL created:", jsonl_path)


✅ JSONL created: C:\Users\dmika\DEV\Projects-local\dp100-learn\data\intel-image-classification\subset\annotations\train_annotations.jsonl


### Local

In [ ]:

# import json

# subset_dir = dataset_dir / "subset"
# jsonl_path = subset_dir / "train_annotations.jsonl"

# records = []

# for class_dir in subset_dir.iterdir():
#     if class_dir.is_dir():
#         label = class_dir.name
#         for img_path in class_dir.glob("*.jpg"):
#             record = {
#                 "image_url": str(img_path.resolve()),  # full path
#                 "label": label
#             }
#             records.append(record)

# # Write to JSONL
# with open(jsonl_path, "w", encoding="utf-8") as f:
#     for r in records:
#         f.write(json.dumps(r) + "\n")

# print(f"✅ JSONL created at: {jsonl_path}")
# print(f"Total records: {len(records)}")

✅ JSONL created at: C:\Users\dmika\DEV\Projects-local\dp100-learn\data\intel-image-classification\subset\train_annotations.jsonl
Total records: 1200


## Create MLTable with the annotations

In [ ]:
%%writefile data/intel-image-classification/subset/annotations/MLTable

paths:
  - file: ./train_annotations.jsonl
transformations:
  - read_json_lines:
        encoding: utf8
        invalid_lines: error
        include_path_column: false
  - convert_column_types:
      - columns: image_url
        column_type: stream_info

Writing data/intel-image-classification/subset//annotations/MLTable


In [47]:
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes

mltable_asset = Data(
    name="intel-image-subset-mltable",
    path=str(annotations_dir),
    type=AssetTypes.MLTABLE,
    datastore=datastore_name,
    description="Intel Image Classification subset prepared for AutoML"
)
registered_mltable = ml_client.data.create_or_update(mltable_asset)
print("✅ Registered MLTable asset:", registered_mltable.id)


Uploading annotations (0.35 MBs): 100%|##########| 348809/348809 [00:00<00:00, 1253593.22it/s]




✅ Registered MLTable asset: /subscriptions/a1267753-4c98-48c1-a8e9-9c7169202ffd/resourceGroups/ml-workspace-dev/providers/Microsoft.MachineLearningServices/workspaces/mlw-dp100-labs/data/intel-image-subset-mltable/versions/1
